# Colab — Unsloth LoRA 학습 (선택 경로)

- **데이터·채팅 형식**: `finetune/train_lora.py` 와 동일 (`instruction` / `input` / `output` → `example_to_text`).
- **스크립트**: `finetune/train_lora_unsloth.py` (기존 `requirements-train.txt` 만으로는 Unsloth 미포함).
- **기존 경로와 공존**: 표준 HF+PEFT만 쓰려면 `colab_train.ipynb` → `train_lora.py` 를 그대로 사용.
- **전체 절차(파일 목록·순서)**: `docs/finetune/colab/COLAB_TRAINING.md`.

**충돌 완화**: 재설치 전 `unsloth` / `unsloth_zoo` 제거 후 아래 셀 순서대로 설치. 팀에서 `--no-deps` 로 고정했다면 두 번째 설치 블록을 사용.

In [ ]:
# [A] 권장: unsloth 가 의존성까지 맞춰 설치 (Colab)
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# [B] 팀 레시피(의존성 수동) — 기존 torch/transformers 와 버전 충돌 날 때만 사용
# %pip uninstall -y unsloth unsloth-zoo
# %pip install --no-deps -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# %pip install --no-deps -q unsloth_zoo
# %pip install -q xformers trl peft accelerate bitsandbytes datasets transformers sentencepiece protobuf
pass

In [ ]:
import os

PROJECT_ROOT = "/content"
FINETUNE_DIR = f"/content"
DATA_PATH = "/content/assembly_style_instruction_dataset_500.jsonl"
OUTPUT_DIR = "/content/lora-output"
BASE_MODEL = "MLP-KTLim/llama-3-Korean-Bllossom-8B"
USE_4BIT = True

# 자동 업로드 옵션
ENABLE_UPLOAD = True
HF_REPO_ID = "seongsu0105/blossom-8b-adapter-unsloth"
HF_REPO_PRIVATE = False

In [ ]:
!pip install -q -r /content/metadata/finetune/requirements-train.txt
!pip install -q python-dotenv huggingface_hub

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

script = Path(FINETUNE_DIR).resolve() / "train_lora_unsloth.py"
if not script.is_file():
    raise FileNotFoundError(f"train_lora_unsloth.py 없음: {script}")

cmd = [
    sys.executable,
    "-u",
    str(script),
    "--data",
    DATA_PATH,
    "--out",
    OUTPUT_DIR,
    "--base-model",
    BASE_MODEL,
    "--epochs",
    "3",
    "--batch-size",
    "2",
    "--grad-accum",
    "8",
    "--max-length",
    "2048",
    "--warmup-steps",
    "50",
    "--save-strategy",
    "steps",
    "--save-steps",
    "100",
]
if USE_4BIT:
    cmd.append("--use-4bit")

tail = deque(maxlen=400)
proc = subprocess.Popen(
    cmd,
    cwd=str(script.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    tail.append(line)
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError("".join(tail))

print(f"학습 완료: {OUTPUT_DIR}")

In [ ]:
if ENABLE_UPLOAD:
    from google.colab import userdata
    from huggingface_hub import HfApi, login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Colab Secrets에 HF_TOKEN을 먼저 등록하세요.")

    if not os.path.isdir(OUTPUT_DIR):
        raise RuntimeError(f"업로드 폴더가 없습니다: {OUTPUT_DIR}")

    login(token=token)
    api = HfApi()
    api.create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        private=HF_REPO_PRIVATE,
        exist_ok=True,
        token=token,
    )
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        token=token,
    )
    print(f"허깅페이스 자동 업로드 완료: https://huggingface.co/{HF_REPO_ID}")
else:
    print("ENABLE_UPLOAD=False 이므로 업로드를 건너뜁니다.")